# Gleich und Wechselstrombrücken

In [287]:
import uncertainties as unc
from uncertainties import ufloat
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import sympy as smp

## Geräte Parameter

### 1.1 Berechnungen

Näherungsformel GLeichung 2.5. Widerstandsinkrement R_1 tauschen mit R_4 da $R_1R_4 = R_2R_3$ gilt



Abgleichbedingung

$$ 
\frac{R_1}{R_2} = \frac{R_3}{R_4}, \quad \frac{R_2}{R_1} = ratio \\
R_3 = R_4 \cdot ratio, \quad R_2 = \frac{R_1}{ratio}
$$

In [288]:
# Gegeben
UH = 3
# Zu Messen
deltaU = 30e-6
deltaR4 = 1
R1 = ufloat(3.5e3,10) # Messergebnis +/- Garantiefehler im MB


In [289]:
def calcR(ratio):
    R4 = (deltaR4 * UH) / (deltaU * (1+1/ratio)*(1+ratio))
    R3 = R4/ratio
    R2 = R1*ratio
    return {
        "Ratio": ratio,
        "R1": R1,
        "R2": R2,
        "R3": R3,
        "R4": R4
    }

data = [calcR(r) for r in [0.5, 1, 2]]
df1 = pd.DataFrame(data, columns=['Ratio', 'R1' ,'R2', 'R3', 'R4'])
print(df1.to_latex(index=False, float_format="%.2f"))
df1


\begin{tabular}{rllrr}
\toprule
Ratio & R1 & R2 & R3 & R4 \\
\midrule
0.50 & 3500+/-10 & 1750+/-5 & 44444.44 & 22222.22 \\
1.00 & 3500+/-10 & 3500+/-10 & 25000.00 & 25000.00 \\
2.00 & 3500+/-10 & 7000+/-20 & 11111.11 & 22222.22 \\
\bottomrule
\end{tabular}



,Ratio,R1,R2,R3,R4
0,0.5,3500+/-10,1750+/-5,44444.444444,22222.222222
1,1.0,3500+/-10,3500+/-10,25000.000000,25000.000000
2,2.0,3500+/-10,7000+/-20,11111.111111,22222.222222


### 1.2 Berechnungen

R2 R3 Grob,
R4 fein,
R1 berechnen

In [322]:
def calcR(ratio, R2, R3, R4):
    R1 = R2*R3/R4
    return {
        "Ratio": ratio,
        "R1": R1,
        "R2": R2,
        "R3": R3,
        "R4": R4
    }

data = [ calcR(r,R2,R3,R4) for r,R2,R3,R4 in [
        (0.5,ufloat(2200,22),   ufloat(47000,470), ufloat(29701,1)),
        (0.5,ufloat(2200,22),   ufloat(22000,220), ufloat(13853,1)),
        (1,  ufloat(4700,47),   ufloat(47000,470), ufloat(63189,1)),
        (1,  ufloat(4700,47),   ufloat(22000,220), ufloat(29476,1)),
        (2,  ufloat(10000,100), ufloat(22000,220), ufloat(63693,1)),
        (2,  ufloat(10000,100), ufloat(10000,100), ufloat(28491,1))
    ]]
df2 = pd.DataFrame(data, columns=['Ratio', 'R1' ,'R2', 'R3', 'R4'])
print(df2.to_latex(
    float_format="{:.2f}".format,
    formatters={
        "R1": "{:.2uS}".format,
        "R2": "{:.2uS}".format,
        "R3": "{:.2uS}".format,
        "R4": "{:.2uS}".format
        }
    ))
df2

\begin{tabular}{lrllll}
\toprule
 & Ratio & R1 & R2 & R3 & R4 \\
\midrule
0 & 0.50 & 3481(49) & 2200(22) & 4.700(47)e+04 & 29701.0(1.0) \\
1 & 0.50 & 3494(49) & 2200(22) & 2.200(22)e+04 & 13853.0(1.0) \\
2 & 1.00 & 3496(49) & 4700(47) & 4.700(47)e+04 & 63189.0(1.0) \\
3 & 1.00 & 3508(50) & 4700(47) & 2.200(22)e+04 & 29476.0(1.0) \\
4 & 2.00 & 3454(49) & 1.000(10)e+04 & 2.200(22)e+04 & 63693.0(1.0) \\
5 & 2.00 & 3510(50) & 1.000(10)e+04 & 1.000(10)e+04 & 28491.0(1.0) \\
\bottomrule
\end{tabular}



,Ratio,R1,R2,R3,R4
0,0.5,(3.48+/-0.05)e+03,2200+/-22,(4.70+/-0.05)e+04,29701.0+/-1.0
1,0.5,(3.49+/-0.05)e+03,2200+/-22,(2.200+/-0.022)e+04,13853.0+/-1.0
2,1.0,(3.50+/-0.05)e+03,(4.70+/-0.05)e+03,(4.70+/-0.05)e+04,63189.0+/-1.0
3,1.0,(3.51+/-0.05)e+03,(4.70+/-0.05)e+03,(2.200+/-0.022)e+04,29476.0+/-1.0
4,2.0,(3.45+/-0.05)e+03,(1.000+/-0.010)e+04,(2.200+/-0.022)e+04,63693.0+/-1.0
5,2.0,(3.51+/-0.05)e+03,(1.000+/-0.010)e+04,(1.000+/-0.010)e+04,28491.0+/-1.0


### 1.3 Empfindlichkeit

In [ ]:
deltaR1 = 9 # 5...10 Ohm
U0min = 30e-6 # Kleinst mögliche spannungsänderung -> Auflösung MB


#### Messung $U_0$

Diese Annäherung gilt für sehr kleine Abweichungen
$$
E = \frac{\partial U_0}{\partial\frac{\Delta R_1}{R_1}} \approx \frac{U_0}{\frac{\Delta R_1}{R_1}}
$$

absolut Kleinste Widerstandsänderung, die am Messgerät erkannt wird

$$
\Delta R_{1,\min} = \frac{\Delta R_1\cdot U_{0,\min}}{U_0}
$$

relativ Kleinste Widerstandsänderung, die am Messgerät erkannt wird

$$
\frac{\Delta R_{1,min}}{R_1} = \frac{\Delta R_1\cdot U_{0,\min}}{U_0\cdot R_1}
$$

In [324]:
# Zu Messen
U0 = [1.86e-3, 1.9e-3, 1.51e-3]

# Berechnung
relR1 = deltaR1/df2["R1"][[0,2,4]]
E = U0/relR1
deltaR1min = [deltaR1*U0min/u for u in U0]
relDeltaR1min = deltaR1min/df2["R1"][[0,2,4]]

data = {
    "E": E,
    "deltaR1min": deltaR1min,
    "relDeltaR1min": relDeltaR1min
}

df3 = pd.DataFrame(data, columns=['E', 'deltaR1min', 'relDeltaR1min'])
print(df3.to_latex(
    float_format="{:.2f}".format,
    formatters={
        "E": "{:.2uS}".format,
        "deltaR1min": "{:.4f}".format,
        "relDeltaR1min": "{:.2uS}".format,
        }
    ))
df3

\begin{tabular}{llrl}
\toprule
 & E & deltaR1min & relDeltaR1min \\
\midrule
0 & 0.719(10) & 0.1452 & 4.170(59)e-05 \\
2 & 0.738(10) & 0.1421 & 4.065(57)e-05 \\
4 & 0.5795(82) & 0.1788 & 5.177(73)e-05 \\
\bottomrule
\end{tabular}



,E,deltaR1min,relDeltaR1min
0,0.719+/-0.010,0.145161,(4.17+/-0.06)e-05
2,0.738+/-0.010,0.142105,(4.06+/-0.06)e-05
4,0.580+/-0.008,0.178808,(5.18+/-0.07)e-05
